## PHASE 4: BRIDGE MINING TO CLASSIFICATION (FEATURE ENGINEERING)
**Duration:** 1 week | **Deliverable:** final_features.csv

### Step 4.1: Convert Mined Patterns to Features

**From Phase 3, you have high-confidence rules. Now create features from them:**

```python
# From Phase 3 top rules, create composite features
df['fraud_risk_score'] = 0  # Initialize

# Rule 1: wire_transfer + upfront_payment
rule_1_match = (df['has_wire_transfer'] == 1) & (df['has_upfront_payment'] == 1)
df.loc[rule_1_match, 'fraud_risk_score'] += 0.87  # confidence from rule

# Rule 2: missing_logo + vague_salary + urgency
rule_2_match = (df['has_missing_logo'] == 1) & \
               (df['has_vague_salary'] == 1) & \
               (df['has_urgency_language'] == 1)
df.loc[rule_2_match, 'fraud_risk_score'] += 0.74

# ... add other top rules ...

# Normalize
df['fraud_risk_score'] = df['fraud_risk_score'] / len(top_rules)
```

**Alternative (Simpler): Count matching patterns:**

```python
# Create feature: "how many red flags does this posting have?"
red_flag_cols = [
    'has_wire_transfer',
    'has_upfront_payment',
    'has_missing_logo',
    'has_vague_salary',
    'has_urgency_language',
    'has_poor_grammar',
    'has_too_good_promise'
]

df['red_flag_count'] = df[red_flag_cols].sum(axis=1)
df['has_multiple_red_flags'] = (df['red_flag_count'] >= 3).astype(int)
```

### Step 4.2: Combine All Features

**Final feature set:**

```python
# Binary structured features (3)
binary_features = ['telecommuting', 'has_company_logo', 'has_questions']

# Categorical features (5) - will be label-encoded
categorical_features = [
    'employment_type',
    'required_experience', 
    'required_education',
    'industry',
    'function'
]

# Text-derived features (from TF-IDF)
tfidf_features = [f'tfidf_{i}' for i in range(200)]

# Red flag features (created in Phase 3)
red_flag_features = [
    'has_wire_transfer',
    'has_upfront_payment',
    'has_missing_logo',
    # ... all red flags ...
]

# Pattern features (from mined patterns)
pattern_features = [
    'fraud_risk_score',
    'red_flag_count'
]

# Combine everything
all_features = binary_features + categorical_features + tfidf_features + red_flag_features + pattern_features

# Final feature dataframe
X = df[all_features]
y = df['fraudulent']
```

---

## PHASE 5: HANDLE CLASS IMBALANCE & PREPARE DATA
**Duration:** 1 week | **Deliverable:** X_train_balanced.csv, X_test.csv, y_train_balanced.csv, y_test.csv

### Step 5.1: Train/Test Split (BEFORE any resampling)

```python
from sklearn.model_selection import train_test_split

# 80/20 split, stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Training set fraud rate: {y_train.mean():.2%}")
print(f"Test set fraud rate: {y_test.mean():.2%}")
# Should both be ~4.8%
```

### Step 5.2: Apply SMOTE to Training Data ONLY

```python
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Original training set size: {len(X_train)}")
print(f"Balanced training set size: {len(X_train_balanced)}")
print(f"Original fraud distribution: {y_train.value_counts().to_dict()}")
print(f"Balanced fraud distribution: {y_train_balanced.value_counts().to_dict()}")
# Test set remains unchanged (real-world distribution)
```

### Step 5.3: Feature Scaling (For tree-based models, optional; for linear models, required)

```python
from sklearn.preprocessing import StandardScaler

# For tree-based models (Random Forest, XGBoost), scaling is optional
# For linear/distance-based, apply scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)  # Use training scaler on test
```


In [1]:
import pandas as pd
import numpy as np
import re
import json
import pickle
from pathlib import Path

# Set up directories
processed_dir = Path('../data/processed')
splits_dir = Path('../data/splits')
models_dir = Path('../models')

splits_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

# Load the cleaned data from Phase 1
df = pd.read_csv(processed_dir / 'cleaned_data.csv')

# Load the top association rules mined in Phase 3
with open(processed_dir / 'mined_fraud_patterns.json', 'r') as f:
    mining_results = json.load(f)
top_rules = pd.DataFrame(mining_results['top_rules'])

print(f"Loaded DataFrame shape: {df.shape}")
print(f"Loaded {len(top_rules)} top association rules.")

Loaded DataFrame shape: (17533, 17)
Loaded 0 top association rules.


In [2]:
# Reconstruct text baseline to generate red flags
text = (
    df['title'].fillna('') + ' ' +
    df['company_profile'].fillna('') + ' ' +
    df['description'].fillna('') + ' ' +
    df['requirements'].fillna('') + ' ' +
    df['benefits'].fillna('')
)

# 1. Re-apply red flags (ensures continuity if Phase 3 didn't overwrite cleaned_data.csv)
text_patterns = {
    'has_wire_transfer': r'wire\s+transfer|wire\s+funds|send\s+money',
    'has_upfront_payment': r'upfront|pay\s+first|advance\s+payment',
    'has_unverified_company': r'\bstartup\b|new\s+company|not\s+established|unknown\s+company',
    'has_generic_location': r'work\s*from\s*home|\bremote\b|\banywhere\b|home\s*based',
    'has_urgency_language': r'\basap\b|urgent|immediate\s+start|now\s+hiring',
    'has_too_good_promise': r'easy\s+money|get\s+rich|quick\s+cash|no\s+experience\s+required',
    'has_data_entry_money': r'data\s+entry.*(commission|payment)|commission.*data\s+entry|paid\s+per\s+entry'
}

for col, pattern in text_patterns.items():
    df[col] = text.str.contains(pattern, case=False, regex=True, na=False).astype(int)

df['has_missing_logo'] = (df['has_company_logo'].isna() | (df['has_company_logo'] == 0)).astype(int)
df['has_vague_salary'] = (df['salary_range'].isna() | df['salary_range'].astype(str).str.strip().str.lower().isin(['not specified', '', 'nan', 'null'])).astype(int)

grammar_pattern = re.compile(r'([!?.,])\1{2,}', re.IGNORECASE)
df['has_poor_grammar'] = (text.apply(lambda x: bool(grammar_pattern.search(x))) | (text.str.count(r'[!?.,;:]').fillna(0) > 80)).astype(int)

red_flag_cols = list(text_patterns.keys()) + ['has_missing_logo', 'has_vague_salary', 'has_poor_grammar']

# --- Step 4.1: Convert Mined Patterns to Features ---

# A. Composite Count Feature
df['red_flag_count'] = df[red_flag_cols].sum(axis=1)
df['has_multiple_red_flags'] = (df['red_flag_count'] >= 3).astype(int)

# B. Calculate Fraud Risk Score from Apriori Rules
df['fraud_risk_score'] = 0.0

for _, rule in top_rules.iterrows():
    conditions = rule['antecedents_str'].split(', ')
    # Create a boolean mask for rows matching all antecedents in the rule
    match = np.ones(len(df), dtype=bool)
    for cond in conditions:
        if cond in df.columns:
            match = match & (df[cond] == 1)
    
    # Add rule confidence to the score for matching rows
    df.loc[match, 'fraud_risk_score'] += rule['confidence']

# Normalize score
if len(top_rules) > 0:
    df['fraud_risk_score'] = df['fraud_risk_score'] / len(top_rules)

print("Engineered Phase 3 pattern features successfully.")
print(df[['red_flag_count', 'fraud_risk_score', 'fraudulent']].head())

C:\Users\hp\AppData\Local\Temp\ipykernel_7248\591491945.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[col] = text.str.contains(pattern, case=False, regex=True, na=False).astype(int)


Engineered Phase 3 pattern features successfully.
   red_flag_count  fraud_risk_score  fraudulent
0               2               0.0           0
1               4               0.0           0
2               1               0.0           0
3               1               0.0           0
4               1               0.0           0


In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Binary Features
binary_features = ['telecommuting', 'has_company_logo', 'has_questions']

# 2. Categorical Features (Label Encoding)
categorical_features = ['employment_type', 'required_experience', 'required_education', 'industry', 'function']
for col in categorical_features:
    le = LabelEncoder()
    df[col] = df[col].astype(str)  # Handle potential NaNs safely
    df[col] = le.fit_transform(df[col])

# 3. Text Features (TF-IDF)
# Dynamically load from Phase 2 if present, otherwise generate them here to prevent pipeline breaks
try:
    tfidf_df = pd.read_csv(processed_dir / 'vectorized_features.csv')
    tfidf_features = [col for col in tfidf_df.columns if col.startswith('tfidf_')]
    for col in tfidf_features:
        df[col] = tfidf_df[col]
    print(f"Successfully loaded {len(tfidf_features)} TF-IDF features from Phase 2.")
except FileNotFoundError:
    print("Phase 2 vectorized data not found. Generating TF-IDF unigrams dynamically...")
    tfidf_vectorizer = TfidfVectorizer(max_features=200, min_df=2, max_df=0.95, stop_words='english')
    tfidf_matrix = tfidf_vectorizer.fit_transform(text)
    tfidf_features = [f'tfidf_{i}' for i in range(tfidf_matrix.shape[1])]
    
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_features)
    # Ensure indexes align
    df = pd.concat([df.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)

# 4. Pattern & Rule Features
pattern_features = ['fraud_risk_score', 'red_flag_count', 'has_multiple_red_flags']

# 5. Final Compilation
all_features = binary_features + categorical_features + tfidf_features + red_flag_cols + pattern_features

# Fill any remaining NaNs in numeric features with 0
df[all_features] = df[all_features].fillna(0)

X = df[all_features]
y = df['fraudulent']

print(f"\nFinal feature matrix shape: {X.shape}")
print(f"Total features extracted: {len(all_features)}")

C:\Users\hp\AppData\Local\Temp\ipykernel_7248\4014895940.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = tfidf_df[col]
C:\Users\hp\AppData\Local\Temp\ipykernel_7248\4014895940.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = tfidf_df[col]
C:\Users\hp\AppData\Local\Temp\ipykernel_7248\4014895940.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(ax

Successfully loaded 350 TF-IDF features from Phase 2.

Final feature matrix shape: (17533, 371)
Total features extracted: 371


C:\Users\hp\AppData\Local\Temp\ipykernel_7248\4014895940.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = tfidf_df[col]
C:\Users\hp\AppData\Local\Temp\ipykernel_7248\4014895940.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = tfidf_df[col]
C:\Users\hp\AppData\Local\Temp\ipykernel_7248\4014895940.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(ax

In [4]:
df = pd.read_csv('../data/processed/final_features.csv')
df.head()

,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,tfidf_ability,tfidf_able,...,has_urgency_language,has_too_good_promise,has_data_entry_money,has_missing_logo,has_vague_salary,has_poor_grammar,fraud_risk_score,red_flag_count,has_multiple_red_flags,fraudulent
0,0,1,0,3,4,6,88,22,0.000000,0.000000,...,0,0,0,0,1,0,0.0,2,0,0
1,0,1,0,1,6,6,75,7,0.025060,0.029916,...,0,0,0,0,1,1,0.0,4,1,0
2,0,1,0,2,7,6,88,23,0.000000,0.000000,...,0,0,0,0,1,0,0.0,1,0,0
3,0,1,0,1,5,1,22,32,0.031105,0.000000,...,0,0,0,0,1,0,0.0,1,0,0
4,0,1,1,1,5,1,51,16,0.000000,0.055022,...,0,0,0,0,1,0,0.0,1,0,0


In [5]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# --- Step 5.1: Train/Test Split ---
# Stratify ensures the 4.8% fraud rate is preserved in both sets before resampling
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Training set fraud rate: {y_train.mean():.2%}")
print(f"Test set fraud rate: {y_test.mean():.2%}")

# --- Step 5.2: Apply SMOTE to Training Data ONLY ---
print("\nApplying SMOTE to training data...")
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Original training set shape: {X_train.shape}")
print(f"Balanced training set shape: {X_train_balanced.shape}")
print(f"Balanced fraud distribution: {y_train_balanced.value_counts().to_dict()}")

# --- Step 5.3: Feature Scaling ---
print("\nApplying StandardScaler...")
scaler = StandardScaler()

# Convert back to DataFrame to preserve column names for feature importance analysis later
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_balanced), columns=all_features)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=all_features)

# --- SAVE EVERYTHING ---
print("\nSaving deliverables to disk...")

# 1. Feature Lists & Models
with open(processed_dir / 'feature_names.pkl', 'wb') as f:
    pickle.dump(all_features, f)
with open(models_dir / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# 2. Master Feature DataFrame
df[all_features + ['fraudulent']].to_csv(processed_dir / 'final_features.csv', index=False)

# 3. Model Training Splits (Unscaled for Trees, Scaled for Linear/SVM)
X_train_balanced.to_csv(splits_dir / 'X_train_balanced.csv', index=False)
X_test.to_csv(splits_dir / 'X_test.csv', index=False)

X_train_scaled.to_csv(splits_dir / 'X_train_scaled.csv', index=False)
X_test_scaled.to_csv(splits_dir / 'X_test_scaled.csv', index=False)

y_train_balanced.to_csv(splits_dir / 'y_train_balanced.csv', index=False)
y_test.to_csv(splits_dir / 'y_test.csv', index=False)

print("Phases 4 & 5 Complete! You are fully prepared to start Phase 6 (Model Building).")

Training set fraud rate: 4.83%
Test set fraud rate: 4.82%

Applying SMOTE to training data...
Original training set shape: (14026, 371)
Balanced training set shape: (26698, 371)
Balanced fraud distribution: {0: 13349, 1: 13349}

Applying StandardScaler...

Saving deliverables to disk...
Phases 4 & 5 Complete! You are fully prepared to start Phase 6 (Model Building).


In [6]:
df.columns.tolist()

['telecommuting',
 'has_company_logo',
 'has_questions',
 'employment_type',
 'required_experience',
 'required_education',
 'industry',
 'function',
 'tfidf_ability',
 'tfidf_able',
 'tfidf_account',
 'tfidf_across',
 'tfidf_activities',
 'tfidf_agency',
 'tfidf_also',
 'tfidf_analysis',
 'tfidf_analytics',
 'tfidf_application',
 'tfidf_applications',
 'tfidf_apply',
 'tfidf_areas',
 'tfidf_around',
 'tfidf_assist',
 'tfidf_available',
 'tfidf_background',
 'tfidf_based',
 'tfidf_believe',
 'tfidf_benefits',
 'tfidf_best',
 'tfidf_big',
 'tfidf_brand',
 'tfidf_brands',
 'tfidf_build',
 'tfidf_building',
 'tfidf_business',
 'tfidf_businesses',
 'tfidf_candidate',
 'tfidf_candidates',
 'tfidf_care',
 'tfidf_career',
 'tfidf_change',
 'tfidf_client',
 'tfidf_clients',
 'tfidf_closely',
 'tfidf_code',
 'tfidf_come',
 'tfidf_communicate',
 'tfidf_communication',
 'tfidf_communications',
 'tfidf_community',
 'tfidf_companies',
 'tfidf_company',
 'tfidf_competitive',
 'tfidf_complex',
 'tfid